In [ ]:
import pandas as pd
import numpy as np
import re
import seaborn as sns
from sklearn.model_selection import train_test_split
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve
from sklearn.preprocessing import StandardScaler

catmap = pd.read_csv("data/catmap.csv")
consDF = pd.read_parquet("data/Consumer.pqt").drop(columns=["credit_score"])
consDF = consDF.dropna(subset=['DQ_TARGET'])

acctDF = pd.read_parquet("data/Account.pqt")
trxnDF = pd.read_parquet("data/Transaction.pqt").drop_duplicates()

acctDF['balance_date'] = pd.to_datetime(acctDF['balance_date'])
trxnDF['posted_date'] = pd.to_datetime(trxnDF['posted_date'])

date_stats = acctDF.groupby('prism_consumer_id')['balance_date'].agg(['min', 'max'])
valid_ids = date_stats[(date_stats['max'] - date_stats['min']).dt.days <= 2].index

consDF = consDF[consDF['prism_consumer_id'].isin(valid_ids)]
acctDF = acctDF[acctDF['prism_consumer_id'].isin(valid_ids)]
trxnDF = trxnDF[trxnDF['prism_consumer_id'].isin(valid_ids)]

trxnDF = trxnDF.merge(catmap, left_on='category', right_on="category_id", how='left').drop(columns=["category_x"])
trxnDF['category'] = trxnDF['category_y'] 
trxnDF = trxnDF.merge(consDF[['prism_consumer_id', 'DQ_TARGET']], on='prism_consumer_id', how='inner')

user_targets_df = trxnDF.groupby('prism_consumer_id')['DQ_TARGET'].max().reset_index()

train_ids, test_ids = train_test_split(
    user_targets_df['prism_consumer_id'],
    test_size=0.2, 
    random_state=42,
    stratify=user_targets_df['DQ_TARGET']
)

test_ids_set = set(test_ids)

test_trxn = trxnDF[trxnDF['prism_consumer_id'].isin(test_ids_set)].copy()
train_trxn = trxnDF[~trxnDF['prism_consumer_id'].isin(test_ids_set)].copy()

test_acct = acctDF[acctDF['prism_consumer_id'].isin(test_ids_set)].copy()
train_acct = acctDF[~acctDF['prism_consumer_id'].isin(test_ids_set)].copy()

def calculate_running_balance(t_df, a_df):
    t_df = t_df.copy()
    t_df['signed_amount'] = np.where(
        t_df['credit_or_debit'] == 'DEBIT', 
        -t_df['amount'], 
        t_df['amount']
    )
    sort_cols = ['prism_consumer_id', 'posted_date']
    if 'prism_transaction_id' in t_df.columns:
        sort_cols.append('prism_transaction_id')
    t_df = t_df.sort_values(sort_cols)
    t_df['raw_cumsum'] = t_df.groupby('prism_consumer_id')['signed_amount'].cumsum()
    snapshot = a_df.sort_values('balance_date').groupby('prism_consumer_id').tail(1)
    snapshot = snapshot.rename(columns={'balance': 'snapshot_balance', 'balance_date': 'snapshot_date'})
    df_merged = t_df.merge(snapshot[['prism_consumer_id', 'snapshot_date', 'snapshot_balance']], on='prism_consumer_id', how='left')
    anchor = df_merged[df_merged['posted_date'] <= df_merged['snapshot_date']].groupby('prism_consumer_id')['raw_cumsum'].last().rename('cumsum_at_snapshot')
    df_merged = df_merged.merge(anchor, on='prism_consumer_id', how='left')
    df_merged['running_balance'] = (df_merged['raw_cumsum'] - df_merged['cumsum_at_snapshot']) + df_merged['snapshot_balance']
    out_cols = [
        'prism_consumer_id', 'posted_date', 'category_id', 'category', 
        'amount', 'signed_amount', 'credit_or_debit', 'running_balance', 'DQ_TARGET'
    ]
    out_cols = [c for c in out_cols if c in df_merged.columns]
    return df_merged[out_cols]

history_train = calculate_running_balance(train_trxn, train_acct)
history_test = calculate_running_balance(test_trxn, test_acct)

def extract_consumer_habits(df):
    df = df.copy()
    df['posted_date'] = pd.to_datetime(df['posted_date'])
    df['day_name'] = df['posted_date'].dt.day_name().str.lower()
    df['day_of_month'] = df['posted_date'].dt.day
    day_avg = df.groupby(['prism_consumer_id', 'category', 'day_name'])['signed_amount'].mean().reset_index()
    weekly_pivot = day_avg.pivot_table(
        index='prism_consumer_id', 
        columns=['category', 'day_name'], 
        values='signed_amount'
    ).fillna(0)
    weekly_pivot.columns = [f"cat_{c}_{d}" for c, d in weekly_pivot.columns]
    dom_avg = df.groupby(['prism_consumer_id', 'category', 'day_of_month'])['signed_amount'].mean().reset_index()
    monthly_pivot = dom_avg.pivot_table(
        index='prism_consumer_id', 
        columns=['category', 'day_of_month'], 
        values='signed_amount'
    ).fillna(0)
    monthly_pivot.columns = [f"cat_{c}_dom_{d}" for c, d in monthly_pivot.columns]

    def get_dual_fft_power(series):
        if len(series) < 14 or series.sum() == 0:
            return 0, 0
        vals = series.values - series.mean()
        fft_mag = np.abs(np.fft.rfft(vals))
        fft_freq = np.fft.rfftfreq(len(vals), d=1)
        weekly_idx = np.argmin(np.abs(fft_freq - (1/7)))
        weekly_power = fft_mag[weekly_idx]
        if len(series) >= 28:
            monthly_idx = np.argmin(np.abs(fft_freq - (1/30)))
            monthly_power = fft_mag[monthly_idx]
        else:
            monthly_power = 0
        return weekly_power, monthly_power

    fft_results = []
    for (cons_id, cat), group in df.groupby(['prism_consumer_id', 'category']):
        daily_series = group.groupby('posted_date')['signed_amount'].sum()
        full_range = pd.date_range(daily_series.index.min(), daily_series.index.max())
        series = daily_series.reindex(full_range, fill_value=0)
        w_power, m_power = get_dual_fft_power(series)
        fft_results.append({
            'prism_consumer_id': cons_id,
            'category': cat,
            'fft_weekly_habit': w_power,
            'fft_monthly_habit': m_power
        })

    fft_df = pd.DataFrame(fft_results)
    if not fft_df.empty:
        fft_pivot_w = fft_df.pivot(index='prism_consumer_id', columns='category', values='fft_weekly_habit').fillna(0)
        fft_pivot_w.columns = [f"cat_{c}_fft_weekly" for c in fft_pivot_w.columns]
        fft_pivot_m = fft_df.pivot(index='prism_consumer_id', columns='category', values='fft_monthly_habit').fillna(0)
        fft_pivot_m.columns = [f"cat_{c}_fft_monthly" for c in fft_pivot_m.columns]
    else:
        fft_pivot_w = pd.DataFrame()
        fft_pivot_m = pd.DataFrame()

    target_map = df.groupby('prism_consumer_id')['DQ_TARGET'].first()
    history_test_df = weekly_pivot.join(monthly_pivot, how='outer')\
                           .join(fft_pivot_w, how='outer')\
                           .join(fft_pivot_m, how='outer')\
                           .join(target_map, how='outer')
    return history_test_df

def build_account_features(acctDF):
    agg_bal = acctDF.groupby("prism_consumer_id", as_index=False).agg(
        total_balance=("balance", "sum"),
        avg_balance=("balance", "mean"),
        max_balance=("balance", "max"),
        min_balance=("balance", "min"),
        std_balance=("balance", "std"),
        num_accounts=("balance", "count"),
    )
    pivot_bal = acctDF.pivot_table(
        index="prism_consumer_id", columns="account_type", values="balance", aggfunc="sum", fill_value=0
    ).reset_index()
    pivot_bal["checking_to_savings_ratio"] = pivot_bal.get("CHECKING", 0) / (pivot_bal.get("SAVINGS", 0) + 1)
    pivot_bal["has_savings_account"] = (pivot_bal.get("SAVINGS", 0) > 0).astype(int)
    keep_cols = ["prism_consumer_id", "checking_to_savings_ratio", "has_savings_account"]
    if "CHECKING" in pivot_bal: keep_cols.append("CHECKING")
    if "SAVINGS" in pivot_bal: keep_cols.append("SAVINGS")
    pivot_bal = pivot_bal[keep_cols]
    acctDF_copy = acctDF.copy()
    acctDF_copy["liquid_component"] = np.where(acctDF_copy["account_type"].isin(["CHECKING", "SAVINGS"]), acctDF_copy["balance"], 0)
    acctDF_copy["liquid_balance"] = acctDF_copy.groupby("prism_consumer_id")["liquid_component"].transform("sum")
    liquidity_flags = acctDF_copy.groupby("prism_consumer_id", as_index=False)["liquid_balance"].first()
    liquidity_flags["low_liquidity"] = (liquidity_flags["liquid_balance"] < 500).astype(int)
    liquidity_flags["is_liquid"] = (liquidity_flags["liquid_balance"] > 0).astype(int)
    return agg_bal, pivot_bal, liquidity_flags

def build_income_features(history_train):
    INCOME_CATEGORIES = ["DEPOSIT", "PAYCHECK", "PAYCHECK_PLACEHOLDER", "INVESTMENT_INCOME", "OTHER_BENEFITS", "UNEMPLOYMENT_BENEFITS", "PENSION"]
    income_txn = history_train[(history_train["credit_or_debit"] == "CREDIT") & (history_train["category"].isin(INCOME_CATEGORIES))].copy()
    income_txn["month"] = income_txn["posted_date"].dt.to_period("M")
    total_income = income_txn.groupby("prism_consumer_id", as_index=False)["amount"].sum().rename(columns={"amount": "total_income"})
    avg_monthly_income = income_txn.groupby(["prism_consumer_id", "month"])["amount"].sum().groupby("prism_consumer_id").mean().reset_index(name="avg_monthly_income")
    income_consistency = income_txn.groupby("prism_consumer_id")["month"].nunique().reset_index(name="num_income_months")
    income_sources = income_txn.groupby("prism_consumer_id")["category"].nunique().reset_index(name="num_income_sources")
    BENEFITS_CATS = ["OTHER_BENEFITS", "UNEMPLOYMENT_BENEFITS", "PENSION"]
    benefits_flag = income_txn[income_txn["category"].isin(BENEFITS_CATS)].groupby("prism_consumer_id").size().reset_index(name="benefits_txn_count")
    benefits_flag["has_benefits_income"] = (benefits_flag["benefits_txn_count"] > 0).astype(int)
    return total_income, avg_monthly_income, income_consistency, income_sources, benefits_flag[["prism_consumer_id", "has_benefits_income"]]

def build_balance_dynamics(history_train):
    df = history_train.sort_values(["prism_consumer_id", "posted_date"]).copy()
    df["daily_balance_change"] = df.groupby("prism_consumer_id")["running_balance"].diff()
    daily_change_features = df.groupby("prism_consumer_id", as_index=False).agg(
        mean_daily_change=("daily_balance_change", "mean"),
        std_daily_change=("daily_balance_change", "std"),
        max_daily_drop=("daily_balance_change", "min"),
        max_daily_increase=("daily_balance_change", "max"),
    )
    latest_dates = df.groupby("prism_consumer_id")["posted_date"].max().reset_index(name="latest_date")
    df = df.merge(latest_dates, on="prism_consumer_id")
    df["date_30d"] = df["latest_date"] - pd.Timedelta(days=30)
    df["date_90d"] = df["latest_date"] - pd.Timedelta(days=90)
    df["date_365d"] = df["latest_date"] - pd.Timedelta(days=365)
    ending_balance = df.groupby("prism_consumer_id")["running_balance"].last()
    def last_bal(date_col):
        filtered = df[df["posted_date"] <= df[date_col]]
        return filtered.groupby("prism_consumer_id")["running_balance"].last()
    recent_balance_features = pd.DataFrame({
        "recent_30d_balance_change": ending_balance - last_bal("date_30d"),
        "recent_90d_balance_change": ending_balance - last_bal("date_90d"),
    }).fillna(0).reset_index()
    mean_balance_features = pd.concat([
        df[df["posted_date"] >= df["date_30d"]].groupby("prism_consumer_id")["running_balance"].mean().rename("mean_balance_30d"),
        df[df["posted_date"] >= df["date_90d"]].groupby("prism_consumer_id")["running_balance"].mean().rename("mean_balance_90d"),
        df[df["posted_date"] >= df["date_365d"]].groupby("prism_consumer_id")["running_balance"].mean().rename("mean_balance_365d"),
    ], axis=1).reset_index()
    return daily_change_features, recent_balance_features, mean_balance_features

def build_balance_magnitude(history_train):
    df = history_train[['prism_consumer_id', 'running_balance']].copy()
    df['abs_running_balance'] = df['running_balance'].abs()
    magnitude_feats = df.groupby('prism_consumer_id')['abs_running_balance'].quantile([0.3, 0.5, 0.8]).unstack()
    magnitude_feats.columns = ['balance_magnitude_30pct', 'balance_magnitude_50pct', 'balance_magnitude_80pct']
    return magnitude_feats.reset_index()

def extract_ALL_features(history_subset, acct_df):
    base_features = extract_consumer_habits(history_subset)
    agg_bal, pivot_bal, liquidity_flags = build_account_features(acct_df)
    income_feats = build_income_features(history_subset)
    balance_feats = build_balance_dynamics(history_subset)
    magnitude_feats = build_balance_magnitude(history_subset)
    
    features_df = base_features.copy()
    all_new_dfs = (agg_bal, pivot_bal, liquidity_flags, *income_feats, *balance_feats, magnitude_feats) 
    for df in all_new_dfs:
        if 'prism_consumer_id' in df.columns:
            df = df.set_index('prism_consumer_id')
        features_df = features_df.join(df, how='left')
    return features_df.fillna(0).reset_index()

print("\n--- PROCESSING TEST SET FEATURES ---")
test_features_df = extract_ALL_features(history_test, acctDF)
X_holdout_base = test_features_df.drop(columns=['DQ_TARGET', 'prism_consumer_id', 'posted_date'], errors='ignore')
y_holdout = test_features_df['DQ_TARGET']

print("\n--- STARTING 20-ITERATION UNDERSAMPLING & FEATURE CREATION ---")
train_targets = history_train.groupby('prism_consumer_id')['DQ_TARGET'].max()
majority_ids = train_targets[train_targets == 0].index
minority_ids = train_targets[train_targets == 1].index
minority_count = len(minority_ids)

best_test_auc = 0
best_model = None
best_model_name = ""
best_iteration = -1
best_test_probs = None
best_y_train_under = None
best_X_train_cols = None

for i in range(20):
    sampled_majority_ids = np.random.choice(majority_ids, size=minority_count, replace=False)
    under_ids = np.concatenate([sampled_majority_ids, minority_ids])
    np.random.shuffle(under_ids)
    
    history_train_under = history_train[history_train['prism_consumer_id'].isin(under_ids)].copy()
    
    print(f"Iteration {i+1:02d} | Extracting features for undersampled subset...")
    train_features_df = extract_ALL_features(history_train_under, acctDF)
    
    X_train = train_features_df.drop(columns=['DQ_TARGET', 'prism_consumer_id', 'posted_date'], errors='ignore')
    y_train = train_features_df['DQ_TARGET']
    
    X_holdout = X_holdout_base.reindex(columns=X_train.columns, fill_value=0)
    
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_holdout_scaled = pd.DataFrame(scaler.transform(X_holdout), columns=X_holdout.columns, index=X_holdout.index)
    
    lr_model = LogisticRegression(max_iter=1000, random_state=42+i, solver='liblinear')
    lr_model.fit(X_train_scaled, y_train)
    
    lr_test_probs = lr_model.predict_proba(X_holdout_scaled)[:, 1]
    lr_test_auc = roc_auc_score(y_holdout, lr_test_probs)
    
    if lr_test_auc > best_test_auc:
        best_test_auc = lr_test_auc
        best_model = lr_model
        best_model_name = "Logistic Regression"
        best_iteration = i + 1
        best_test_probs = lr_test_probs
        best_y_train_under = y_train
        best_train_probs = lr_model.predict_proba(X_train_scaled)[:, 1]
        best_X_train_cols = X_train.columns
    
    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.5,
        max_delta_step=2, 
        random_state=42+i,
        n_jobs=-1,
        early_stopping_rounds=50
    )
    
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_holdout, y_holdout)],
        verbose=False
    )
    
    xgb_test_probs = xgb_model.predict_proba(X_holdout)[:, 1]
    xgb_test_auc = roc_auc_score(y_holdout, xgb_test_probs)
    
    if xgb_test_auc > best_test_auc:
        best_test_auc = xgb_test_auc
        best_model = xgb_model
        best_model_name = "XGBoost"
        best_iteration = i + 1
        best_test_probs = xgb_test_probs
        best_y_train_under = y_train
        best_train_probs = xgb_model.predict_proba(X_train)[:, 1]
        best_X_train_cols = X_train.columns

    print(f"Iteration {i+1:02d} | LR AUC: {lr_test_auc:.4f} | XGB AUC: {xgb_test_auc:.4f}\n")

print(f"\n--- BEST MODEL FOUND: {best_model_name} (Iteration {best_iteration}) ---")
print(f"Train ROC AUC (Undersampled): {roc_auc_score(best_y_train_under, best_train_probs):.4f}")
print(f"Test ROC AUC (Holdout):       {best_test_auc:.4f}")

fpr, tpr, thresholds = roc_curve(best_y_train_under, best_train_probs)
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_threshold = thresholds[best_idx]

roc_preds = (best_test_probs >= best_threshold).astype(int)

print(f"\n--- OPTIMAL ROC THRESHOLD SET OUTCOMES (Threshold = {best_threshold:.4f}) ---")
print(classification_report(y_holdout, roc_preds))

target_threshold = 0.20 
custom_preds = (best_test_probs >= target_threshold).astype(int)

print(f"\n--- CUSTOM BUSINESS THRESHOLD OUTCOMES (Threshold = {target_threshold:.2f}) ---")
print(classification_report(y_holdout, custom_preds))

if best_model_name == "XGBoost":
    importances = best_model.feature_importances_
    x_label = 'Gain (Feature Importance)'
else:
    importances = np.abs(best_model.coef_[0])
    x_label = 'Absolute Coefficient Magnitude'

importance_df = pd.DataFrame({
    'Feature': best_X_train_cols,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importance_df.head(20), x='Importance', y='Feature', hue='Feature', palette='viridis', legend=False)
plt.title(f'Top 20 Predictors ({best_model_name})')
plt.xlabel(x_label)
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


--- PROCESSING TEST SET FEATURES ---

--- STARTING 20-ITERATION UNDERSAMPLING & FEATURE CREATION ---
Iteration 01 | Extracting features for undersampled subset...


/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kylechoi/Desktop/Desktop - kyle의 MacBook

Iteration 01 | LR AUC: 0.6109 | XGB AUC: 0.7958

Iteration 02 | Extracting features for undersampled subset...


/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Iteration 02 | LR AUC: 0.5925 | XGB AUC: 0.8019

Iteration 03 | Extracting features for undersampled subset...


...

Iteration 10 | LR AUC: 0.6239 | XGB AUC: 0.8011

Iteration 11 | Extracting features for undersampled subset...


/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Iteration 11 | LR AUC: 0.6047 | XGB AUC: 0.7944

Iteration 12 | Extracting features for undersampled subset...


/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kylechoi/Desktop/Desktop - kyle의 MacBook Pro/DSC180B/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Iteration 12 | LR AUC: 0.6364 | XGB AUC: 0.7985

Iteration 13 | Extracting features for undersampled subset...


In [ ]:
from sklearn.metrics import roc_curve, classification_report
import matplotlib.pyplot as plt
import numpy as np

# 1. Calculate Optimal Threshold using Youden's J statistic on the Training Set
# J = True Positive Rate (Recall) - False Positive Rate
fpr, tpr, thresholds = roc_curve(y_train, best_train_probs)
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_threshold = thresholds[best_idx]

print(f"--- THRESHOLD OPTIMIZATION (Youden's J) ---")
print(f"Optimal ROC Threshold: {best_threshold:.4f}\n")

# 2. Apply the new threshold to the Holdout Test Set
roc_test_preds = (xgb_test_probs >= best_threshold).astype(int)

# 3. Print the True Adjusted Outcomes
print("--- ADJUSTED FINAL TEST SET OUTCOMES ---")
print(classification_report(y_holdout, roc_test_preds))

# 4. Visualize the Probability Distributions
plt.figure(figsize=(10, 6))
# Using density=True normalizes the histograms so we can compare the shapes fairly
plt.hist(xgb_test_probs[y_holdout == 0], bins=50, alpha=0.6, label='Actual Good (0)', color='green', density=True)
plt.hist(xgb_test_probs[y_holdout == 1], bins=50, alpha=0.6, label='Actual Bad (1)', color='red', density=True)

# Draw the threshold line
plt.axvline(best_threshold, color='black', linestyle='dashed', linewidth=2, label=f'Threshold ({best_threshold:.3f})')

plt.title('Test Set Predicted Probability Distributions')
plt.xlabel('Predicted Probability of Delinquency')
plt.ylabel('Density')
plt.legend()
plt.show()